# Forge: a short campaign

The same window and covariance as `block.ipynb`, a `Wheel` with that
covariance registered as a `FixedNoiseBlock` noise block, the block registered
as a signal block on its grid, and a campaign file the Wheel writes itself:
every block's parameters at registration and after each cycle, read back
through `wheel.l2` (and `L2Data.open` once the campaign is over). A second block goes where the comment says. The block
starts from a prior draw, so the residual's drop over the first cycle is the
block discarding that start, not a fit: compare with the data's own rms.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import tempfile

import numpy as np

from block import ForgeBlock
from data import analytic_covariance, load_window, white_psd
from enchilada import FixedNoiseBlock, Wheel, transform

MOJITO_PATH = os.environ.get("ENCHILADA_MOJITO_L1")
if not MOJITO_PATH:
    raise SystemExit(
        "set ENCHILADA_MOJITO_L1 to a Mojito-layout L1 file "
        "(in Jupyter: %env ENCHILADA_MOJITO_L1=/path/to/file.h5, then rerun this cell)"
    )
DURATION_S = float(os.environ.get("FORGE_DURATION_S", 86400.0))
FACTOR = int(os.environ.get("FORGE_FACTOR", 10))
DOMAIN = "frequency"  # the grid ForgeBlock renders on; see _check_residual
# DOMAIN = "wdm" needs the row count: it and the decimated window length divided
# by it must both be even integers (64 rows for the default day at factor 10
# from a 4 Hz file).
NUM_FREQUENCY_DIVISIONS = None
CHANNELS = ("A", "E")
TRUTH = {}  # EDIT: injected parameter values, when you know them: {"amplitude": 1e-18}

window = load_window(
    MOJITO_PATH, factor=FACTOR, duration_s=DURATION_S, channels=CHANNELS
)
# A white stand-in for the simulation's noise model, at the window's own level
# (for white native noise about 12 % below the in-band PSD: decimation rolls
# off the top fifth of the band; a loud signal in the window inflates it).
# The GB example's `FixedLISANoise().psd` (copy `examples/gb_model.py` beside
# this notebook; needs lisaanalysistools) is a TDI 1.5 sensitivity; Mojito's
# channels are TDI 2.0, so convert it before using it as `PSD`.
LEVEL = 2 * np.mean([window.channel_data[name].var() for name in CHANNELS]) / window.fs
PSD = white_psd(LEVEL)
residual = transform(window, DOMAIN, num_frequency_divisions=NUM_FREQUENCY_DIVISIONS)
covariance = analytic_covariance(residual, PSD)

In [ ]:
def rms(data):
    return float(
        np.sqrt(np.mean([np.abs(data.channel_data[name]) ** 2 for name in CHANNELS]))
    )


print(f"data rms                      {rms(residual):.3e}")
# EDIT: a durable path for a real campaign. The Wheel refuses an existing
# file, so a fresh temporary directory per run keeps this cell re-runnable.
OUTPUT_PATH = os.path.join(tempfile.mkdtemp(), "forge.h5")
wheel = Wheel(residual, random_seed=0, output_path=OUTPUT_PATH)
# The PSD's covariance enters as a noise block, registered first so the signal
# blocks' starts receive it.
wheel.add(FixedNoiseBlock("psd", covariance), role="noise")
block = ForgeBlock(name="forge", frequency_hz=1e-3)  # EDIT: your block's configuration
wheel.add(block, role="signal", data_domain=DOMAIN)
# A second signal block goes here, on the covariance's grid:
# wheel.add(OtherBlock(name="other"), role="signal", data_domain=DOMAIN)
# A signal block receives the sum of the noise blocks' covariances, which lives
# on one grid, so every block registers on that grid (a covariance does not
# translate between grids); to work on another grid, move DOMAIN, and with it
# the covariance, for the whole campaign.
print(f"residual rms after the starts {rms(wheel.residual()):.3e}")

In [ ]:
NUM_CYCLES = 200
WARM_UP = 0  # the placeholder draws the posterior exactly; an MCMC sampler needs some
wheel.run(NUM_CYCLES)
# The file holds every block's parameters: entry 0 is the start, then one
# entry per cycle. chain() returns them as one array per parameter.
for key in wheel.l2.parameters(block.name):
    samples = wheel.l2.chain(block.name, key)[1 + WARM_UP :]
    if samples.ndim != 1 or samples.dtype.kind not in "biuf":
        continue  # summarise real scalars only
    print(
        f"posterior {key}: {samples.mean():+.3e} +/- {samples.std():.3e} "
        f"over {samples.size} cycles"
    )
    if key in TRUTH:
        offset = (samples.mean() - TRUTH[key]) / samples.std()
        print(
            f"truth {TRUTH[key]:+.3e}: the posterior mean is {offset:+.2f} sigma away"
        )
print(f"residual rms after the run    {rms(wheel.residual()):.3e}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is not None:
    keys = [
        key
        for key in wheel.l2.parameters(block.name)
        if wheel.l2.chain(block.name, key).ndim == 1
    ]
    figure, axes = plt.subplots(
        len(keys), 1, figsize=(8, 3 * len(keys)), sharex=True, squeeze=False
    )
    for axis, key in zip(axes[:, 0], keys, strict=True):
        axis.plot(wheel.l2.chain(block.name, key)[1:])
        axis.axvline(WARM_UP, color="k", linestyle=":")
        if key in TRUTH:
            axis.axhline(TRUTH[key], color="k", linestyle="--")
        axis.set_ylabel(key)
    axes[-1, 0].set_xlabel("cycle")
    plt.show()

## Next

- Replace the placeholder model in `block.py` and rerun both notebooks.
- Model the noise with a noise block (`role="noise"`), registered before the
  signal blocks whose starts need its covariance; blocks start in registration
  order. It publishes on the campaign's one grid and its covariance adds to
  the noise sum beside the PSD's `FixedNoiseBlock`; drop that block when yours
  models the whole noise.
- Run independent campaigns with fresh block instances and different seeds.
- Checkpointing and restarts are outside the forge.